# Minería de Datos — Sesión 8
## Normalización y estandarización

**26150 · grupo 020-83 · jueves 3 de septiembre de 2026**
**Bloque 2 — Preparación de datos**

---

Un dataset con `edad` en años (17 a 60) e `ingreso` en pesos (0 a 4 000 000). Las dos son números,
pero **no están en la misma escala** — y hay algoritmos para los que eso no es un detalle estético
sino un error de resultado.

Hoy: **cuándo importa la escala, qué transformación aplicar, y el error de procedimiento que
invalida un proyecto entero.** Ese último punto —la fuga de información— es lo que más se cobra en
el informe final.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 110)
rng = np.random.default_rng(2026)
print("pandas", pd.__version__)

In [ ]:
n = 600
jornada = rng.choice(["Diurna", "Nocturna"], n, p=[0.6, 0.4])
edad = np.where(jornada == "Nocturna", rng.normal(28, 6, n),
                rng.normal(21, 3, n)).round().clip(17, 60)
horas = np.where(jornada == "Nocturna", rng.normal(38, 8, n),
                 rng.normal(9, 7, n)).round().clip(0, 60)
ingreso = (380_000 + 24_000 * horas + 11_000 * edad
           + rng.normal(0, 180_000, n)).round(-3).clip(0, None)

# un par de ingresos atipicos: dos estudiantes que ademas tienen negocio
ingreso[rng.choice(n, 6, replace=False)] *= 6

df = pd.DataFrame({"jornada": jornada, "edad": edad,
                   "horas": horas, "ingreso": ingreso})
df.describe().round(1)

---

## 1. Por qué importa: la distancia

Casi todo lo que viene en el curso se apoya en **medir qué tan parecidos son dos registros**. Y
«parecido» casi siempre significa distancia euclidiana:

$$d(\mathbf{a},\mathbf{b}) = \sqrt{\sum_{j=1}^{p} (a_j - b_j)^2}$$

El problema está en que esa suma **no sabe de unidades**.

In [ ]:
a = df.loc[0, ["edad", "horas", "ingreso"]].astype(float)
b = df.loc[1, ["edad", "horas", "ingreso"]].astype(float)

dif = (a - b).abs()
d2 = (a - b) ** 2
print(pd.DataFrame({"|diferencia|": dif.round(0),
                    "aporte a d²": d2.round(0),
                    "% del total": (d2 / d2.sum() * 100).round(2)}))

> **Ese porcentaje es toda la lección.** `ingreso` se lleva prácticamente el 100 % de la distancia.
> `edad` y `horas` están en la fórmula pero **no participan de la decisión**.
>
> No es que el ingreso sea más importante: es que se mide en unidades más grandes. Un modelo basado
> en distancias, sobre estos datos sin escalar, **es un modelo de una sola variable**.

### A quién le cambia el desempeño y a quién no

| El desempeño cambia | Por qué |
|---|---|
| **KNN**, k-medias, DBSCAN | Miden distancias. Es el caso de arriba |
| **SVM** | El margen se define en el espacio de las variables |
| **PCA** | Maximiza varianza; sin escalar, gana la variable de unidades más grandes |
| **Redes neuronales** | Escalas dispares hacen que el descenso de gradiente converja mal |
| Regresión **regularizada** (ridge, lasso) | La penalización castiga coeficientes grandes, y el tamaño del coeficiente depende de la escala |

| El desempeño **no** cambia | Por qué |
|---|---|
| **Árboles de decisión** | Solo preguntan `¿x < umbral?`. Reescalar mueve el umbral, no el corte |
| **Bosques aleatorios**, *gradient boosting* | Son árboles |
| Regresión lineal **sin** regularización | Los coeficientes se ajustan solos a las unidades |

> **Ojo con leer esta tabla como una autorización para no escalar.** Dice qué modelos cambian
> **de desempeño**, y eso es solo la mitad del asunto: el escalado también decide **si el resultado
> se puede leer**. Volvemos sobre esto en la sección 7 bis, y ahí se entiende por qué en este curso
> se escala siempre.

---

## 2. Mín-máx

$$x' = \frac{x - \min(x)}{\max(x) - \min(x)}$$

Lleva todo al rango $[0, 1]$. El mínimo va a 0, el máximo a 1, el resto se reparte proporcionalmente.

In [ ]:
num = ["edad", "horas", "ingreso"]
X = df[num]

minmax = (X - X.min()) / (X.max() - X.min())
minmax.describe().round(3).loc[["min", "max", "mean", "50%"]]

**A favor:** rango garantizado — útil cuando el algoritmo lo exige (redes con activación sigmoide,
imágenes).

**En contra, y es grave:** el mínimo y el máximo son **los dos valores más frágiles del dataset**.
Un solo atípico define el máximo y **aplasta todo lo demás**.

In [ ]:
print("mediana de ingreso ya escalado:", round(minmax["ingreso"].median(), 3))
print("percentil 90            :", round(minmax["ingreso"].quantile(0.90), 3))
print("\nLos seis atipicos empujaron el maximo, y el 90 % de los datos")
print("quedo comprimido en la parte baja del rango.")

### Y el problema serio: un valor nuevo se sale del rango

Si mañana llega un registro con un ingreso mayor que el máximo visto, su valor escalado es **mayor
que 1**. La garantía del rango solo vale para los datos con los que se calculó.

---

## 3. Puntuación z (estandarización)

$$z = \frac{x - \bar{x}}{s}$$

Deja media 0 y desviación estándar 1. **No acota el rango**: dice a cuántas desviaciones está cada
valor de su media.

In [ ]:
z = (X - X.mean()) / X.std()
print(z.describe().round(3).loc[["mean", "std", "min", "max"]])

In [ ]:
# Verificar que ahora las tres variables SI participan de la distancia
az, bz = z.loc[0], z.loc[1]
d2z = (az - bz) ** 2
print(pd.DataFrame({"aporte a d²": d2z.round(3),
                    "% del total": (d2z / d2z.sum() * 100).round(1)}))

> **Comparar con la tabla del principio.** Ahora las tres variables aportan a la distancia. Eso es
> exactamente lo que se buscaba.

**A favor:** es la opción por defecto. Resiste mucho mejor los atípicos que mín-máx, y un valor
nuevo fuera de rango simplemente da un z grande — no rompe nada.

**En contra:** no acota. Y la media y la desviación **tampoco son inmunes** a los atípicos: los seis
ingresos multiplicados por 6 subieron la media y la desviación de todos.

---

## 4. Escalado robusto

$$x' = \frac{x - \text{mediana}(x)}{\text{IQR}(x)}$$

La misma idea de la z, pero con **mediana** en vez de media e **IQR** en vez de desviación — las dos
medidas que ya vimos que no se mueven con los atípicos.

In [ ]:
mediana = X.median()
iqr = X.quantile(0.75) - X.quantile(0.25)
rob = (X - mediana) / iqr

comparacion = pd.DataFrame({
    "z: |valor| máximo": z.abs().max().round(2),
    "robusto: |valor| máximo": rob.abs().max().round(2),
    "z: rango del 90 % central": (z.quantile(0.95) - z.quantile(0.05)).round(2),
    "robusto: rango del 90 % central": (rob.quantile(0.95) - rob.quantile(0.05)).round(2),
})
print(comparacion)

> **Lo que hay que leer ahí:** con el escalado robusto los atípicos **siguen siendo extremos** —no
> se ocultan—, pero **el grueso de los datos queda mejor repartido**, porque el divisor no se infló.
>
> Con la z, los seis atípicos inflaron `s`, y eso **encogió a todos los demás** hacia el centro.

---

## 5. Las tres, mirándolas

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(13, 3.4))
for a, (dat, tit) in zip(ax, [(X["ingreso"], "original"),
                              (minmax["ingreso"], "mín-máx"),
                              (z["ingreso"], "puntuación z"),
                              (rob["ingreso"], "robusto")]):
    a.hist(dat, bins=40, color="#4C7A9E", edgecolor="white", linewidth=0.4)
    a.set_title(tit, fontsize=10)
    a.set_yticks([])
plt.tight_layout()
plt.show()

> **La forma no cambia.** Las cuatro son la misma distribución con otros números en el eje.
>
> Escalar **no arregla** la asimetría, ni la bimodalidad, ni los atípicos. Solo cambia las unidades.
> Si el problema es la forma, la herramienta es otra: logaritmo, raíz, o dejarla como está.

### Cuando la forma sí es el problema

In [ ]:
log_ing = np.log1p(X["ingreso"])          # log(1 + x): tolera el cero

fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
ax[0].hist(X["ingreso"], bins=40, color="#B0442F", edgecolor="white", linewidth=0.4)
ax[0].set_title("ingreso — cola larga a la derecha", fontsize=10)
ax[1].hist(log_ing, bins=40, color="#1E7F63", edgecolor="white", linewidth=0.4)
ax[1].set_title("log(1 + ingreso) — casi simétrica", fontsize=10)
for a in ax: a.set_yticks([])
plt.tight_layout()
plt.show()

> **Orden correcto cuando hay asimetría fuerte:** primero el logaritmo (arregla la forma), después la
> puntuación z (arregla la escala). No al revés — el logaritmo de un valor negativo no existe.

---

## 6. La fuga de información

Todo lo anterior es fácil. **Esto es lo que se hace mal, y lo que invalida un proyecto.**

Escalar usa dos números calculados de los datos: mínimo y máximo, o media y desviación. La pregunta
es **de cuáles datos**.

In [ ]:
from sklearn.model_selection import train_test_split

# ---------- LO INCORRECTO ----------
X_mal = (df[num] - df[num].mean()) / df[num].std()     # escalar TODO primero
tr_mal, te_mal = train_test_split(X_mal, test_size=0.3, random_state=7)

# ---------- LO CORRECTO ----------
tr, te = train_test_split(df[num], test_size=0.3, random_state=7)
mu, sd = tr.mean(), tr.std()          # los parametros salen SOLO de entrenamiento
tr_ok = (tr - mu) / sd
te_ok = (te - mu) / sd                # y se APLICAN a prueba

print("media de prueba, forma incorrecta:", te_mal.mean().round(4).to_dict())
print("media de prueba, forma correcta  :", te_ok.mean().round(4).to_dict())

**Mire las dos líneas.** En la forma incorrecta, la media del conjunto de prueba da prácticamente 0
— porque **se centró usando información que incluía a prueba**.

En la forma correcta no da 0, y **eso es lo que se busca**: el conjunto de prueba es un dato que el
modelo no ha visto. Si su media da exactamente 0, es porque participó en el cálculo.

> **Por qué importa de verdad.** El conjunto de prueba existe para estimar cómo se comporta el modelo
> con datos que nunca vio. Si la media de prueba entró al escalado, el modelo ya recibió información
> de esos datos: **la estimación queda optimista** y el modelo rinde peor en producción de lo que
> decía el informe.
>
> Es un error silencioso. **No lanza ninguna advertencia**, y el resultado se ve mejor. Por eso se
> revisa por procedimiento, no por síntoma.

### La regla, en una línea

**`fit` solo con entrenamiento. `transform` con todo.**

In [ ]:
from sklearn.preprocessing import StandardScaler

esc = StandardScaler()
tr_sk = esc.fit_transform(tr)     # fit_transform SOLO en entrenamiento
te_sk = esc.transform(te)         # transform (sin fit) en prueba

print("¿coincide con el calculo a mano?",
      bool(np.allclose(tr_sk, tr_ok.values, atol=1e-8)))

> `StandardScaler` divide por la desviación poblacional y `pandas` por la muestral, así que puede
> haber una diferencia mínima. La estructura es idéntica: **`fit` en entrenamiento, `transform` en
> ambos**.

---

## 7. El `Pipeline`: que el error no se pueda cometer

La forma robusta de no equivocarse **no es acordarse**: es usar una estructura donde el error sea
imposible.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsRegressor

y = df["ingreso"]
Xf = df[["edad", "horas"]]
Xtr, Xte, ytr, yte = train_test_split(Xf, y, test_size=0.3, random_state=7)

modelo = Pipeline([
    ("imputar", SimpleImputer(strategy="median")),
    ("escalar", StandardScaler()),
    ("knn", KNeighborsRegressor(n_neighbors=5)),
])

modelo.fit(Xtr, ytr)      # ajusta imputador y escalador SOLO con entrenamiento
print("R² en prueba:", round(modelo.score(Xte, yte), 3))

---

## 6 bis. ¿Y cuánto cambia de verdad el resultado?

Todo lo anterior es un argumento: *«sin escalar, `ingreso` domina la distancia»*. Suena convincente,
pero **no lo hemos medido**. Vamos a medirlo, porque un argumento que no se comprueba es una opinión.

El experimento es honesto: **el mismo modelo, la misma partición, la misma semilla.** Lo único que
cambia es si se escala.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

y  = df["ingreso"]
Xf = df[["edad", "horas"]].copy()
Xf["ruido_grande"] = rng.normal(500_000, 200_000, len(df))   # variable inútil, escala enorme

Xtr, Xte, ytr, yte = train_test_split(Xf, y, test_size=0.3, random_state=7)

def evaluar(modelo, escalar):
    pasos = ([("escalar", StandardScaler())] if escalar else []) + [("m", modelo)]
    return Pipeline(pasos).fit(Xtr, ytr).score(Xte, yte)

filas = []
for nombre, mod in [("KNN (distancias)", KNeighborsRegressor(n_neighbors=5)),
                    ("Árbol (umbrales)", DecisionTreeRegressor(random_state=7, max_depth=6))]:
    sin = evaluar(mod, False)
    con = evaluar(mod, True)
    filas.append({"modelo": nombre, "sin escalar": round(sin, 3),
                  "escalado": round(con, 3), "diferencia": round(con - sin, 3)})

print(pd.DataFrame(filas).to_string(index=False))

> **Ahí está la demostración, y son dos resultados en uno.**
>
> - **El KNN mejora mucho.** Sin escalar, `ruido_grande` —una variable que no explica nada, pero que
>   vale cientos de miles— se lleva casi toda la distancia, y el modelo termina buscando vecinos
>   parecidos *en ruido*. Escalar la pone en su lugar.
> - **El árbol no se mueve.** Y no es casualidad ni margen de error: el árbol solo pregunta
>   `¿x < umbral?`. Reescalar mueve el umbral, no el corte, así que **el árbol resultante es el
>   mismo**.
>
> Eso convierte la tabla de la sección 1 en un hecho medido.

**Una advertencia sobre este experimento:** metí `ruido_grande` a propósito para que el efecto se
viera grande. Con variables reales la diferencia suele ser menor, pero **va en la misma dirección**.
Lo que no cambia nunca es el árbol.

> **Cuidado con la conclusión fácil.** Es tentador cerrar aquí diciendo *«si uso árboles, no escalo»*.
> El desempeño del árbol efectivamente no cambia — pero eso **no es todo lo que el escalado
> decide**. La sección que sigue muestra la otra mitad del asunto.

> **Lo que hace el `Pipeline`:** al llamar `fit`, ajusta cada paso **solo con entrenamiento**. Al
> llamar `score` o `predict` sobre prueba, aplica los parámetros ya aprendidos.
>
> **La fuga se vuelve imposible sin tener que acordarse de nada.** Por eso el proyecto de este curso
> se entrega con `Pipeline`, y no con transformaciones sueltas.

---

## 8. Cómo se decide

```
1 · ¿Hay asimetría fuerte?  ──→ logaritmo PRIMERO (arregla la forma)

2 · ¿Qué transformación de escala?
     ├── ¿hay atípicos que no se van a quitar? ──→ escalado robusto
     ├── ¿el algoritmo exige rango acotado?    ──→ mín-máx
     └── en cualquier otro caso ───────────────→ puntuación z

3 · SIEMPRE: partir primero · fit con entrenamiento · transform con ambos
     └── o mejor: Pipeline, y el error se vuelve imposible
```

> **Antes se decía aquí que si el modelo era de árboles no valía la pena escalar.** Eso es cierto
> **solo para el desempeño**. Como acabamos de ver en la sección 7 bis, el escalado también decide
> **si los coeficientes de un modelo lineal se pueden leer** — y en este curso se comparan varios
> modelos, no uno solo. **Por eso se escala siempre.**

---

## 9. Trabajo de la sesión — Taller 4

Sobre **su** dataset. El enunciado completo está en `TALLER_4_MD_2026-3.md`; el resumen:

1. **La tabla del aporte a la distancia**, antes de escalar: qué variable domina y en qué porcentaje.
2. **Las tres transformaciones** aplicadas a la variable de mayor rango, con los cuatro histogramas.
   ¿Cambió la forma?
3. **Los tres modelos** —KNN, árbol y Ridge—, cada uno **con y sin escalado**. Una tabla de seis
   filas, con la métrica justificada.
4. **La lectura de esa tabla**: una frase por modelo, explicando **el resultado obtenido**, no el
   esperado.
5. **La fuga de información**, mostrada en sus datos, y el `Pipeline` que la vuelve imposible.
6. **¿Cuál sabe explicarse?** Coeficientes de Ridge escalado contra sin escalar · importancias y
   reglas del árbol · qué es lo único que puede mostrar el KNN.

---

## Cierre

- **La escala decide quién manda en la distancia.** Sin escalar, un modelo de distancias es un
  modelo de la variable de unidades más grandes.
- **Mín-máx** acota pero es frágil · **z** es el defecto · **robusto** cuando hay atípicos.
- **Escalar no cambia la forma.** Para eso está el logaritmo.
- **La fuga de información no da error, da un resultado mejor de lo real.** `fit` solo con
  entrenamiento.
- **El escalado también hace legible el resultado.** Sin él, los coeficientes de un modelo lineal
  no se pueden comparar entre sí.

**Sábado 5:** codificación de categóricas —one-hot, ordinal, alta cardinalidad— y
`ColumnTransformer` para tratar numéricas y categóricas en un mismo flujo. **Cierra el bloque 2.**

**Jueves 10:** arrancan los modelos en serio, con árboles de decisión.

**Taller 4** — cierra el **domingo 6 de septiembre, 11:59 p. m.**

In [ ]:
from sklearn.linear_model import Ridge

modelos = [
    ("KNN",   KNeighborsRegressor(n_neighbors=5)),
    ("Arbol", DecisionTreeRegressor(random_state=7, max_depth=6)),
    ("Ridge", Ridge(alpha=1.0)),
]

filas = []
for nombre, mod in modelos:
    for escalar in (False, True):
        r2 = evaluar(mod, escalar)
        filas.append({"modelo": nombre,
                      "escalado": "sí" if escalar else "no",
                      "R2 en prueba": round(r2, 3)})

tabla = pd.DataFrame(filas)
print(tabla.to_string(index=False))

print("\ndiferencia por modelo (escalado - sin escalar):")
for nombre, _ in modelos:
    sub = tabla[tabla["modelo"] == nombre].set_index("escalado")["R2 en prueba"]
    print(f"  {nombre:6s} {sub['sí'] - sub['no']:+.3f}")

> **Tres comportamientos distintos, y hay que leerlos uno por uno:**
>
> - **KNN sube.** Se apoya en distancias, y sin escalar la variable de unidades grandes se las lleva.
> - **El árbol no se mueve.** Reescalar desplaza el umbral, no el corte: el árbol resultante es el
>   mismo.
> - **Ridge apenas cambia en desempeño.** Y aquí viene lo importante: **eso no significa que dé
>   igual escalarlo.**

### Por qué Ridge sí necesita escalado, aunque el R² no lo note

Un modelo lineal aprende **un coeficiente por variable**. Ese coeficiente dice *cuánto cambia la
predicción cuando la variable sube una unidad* — y «una unidad» significa algo distinto en cada
variable.

Miremos qué pasa al intentar leer los coeficientes en los dos casos.

In [ ]:
sin_esc = Ridge(alpha=1.0).fit(Xtr, ytr)
con_esc = Pipeline([("escalar", StandardScaler()),
                    ("ridge", Ridge(alpha=1.0))]).fit(Xtr, ytr)

comp = pd.DataFrame({
    "variable": Xtr.columns,
    "coef SIN escalar": sin_esc.coef_.round(2),
    "coef CON escalar": con_esc.named_steps["ridge"].coef_.round(0),
})
print(comp.to_string(index=False))

print("\nOrden de importancia segun cada lista:")
print("  sin escalar:", list(comp.reindex(comp['coef SIN escalar'].abs()
                                          .sort_values(ascending=False).index)["variable"]))
print("  con escalar:", list(comp.reindex(comp['coef CON escalar'].abs()
                                          .sort_values(ascending=False).index)["variable"]))

> **Los dos órdenes son distintos, y uno de los dos miente.**
>
> **Sin escalar**, el coeficiente de `edad` se compara contra el de `ruido_grande`, que se mide en
> cientos de miles. Un coeficiente pequeño puede corresponder a la variable más influyente, solo
> porque sus unidades son grandes. **Ese orden no dice nada.**
>
> **Con escalar**, todas las variables están en desviaciones estándar, así que «una unidad» significa
> lo mismo en todas. Ahí sí el coeficiente más grande es el de la variable que más mueve la
> predicción.

**Ésta es la razón por la que todos escalan, incluso quien vaya con árboles:** el escalado no es solo
una preparación para que el modelo funcione. **Es lo que hace legible el resultado.**

### Y el árbol, ¿qué dice?

In [ ]:
from sklearn.tree import export_text

arbol = DecisionTreeRegressor(random_state=7, max_depth=3).fit(Xtr, ytr)

print("Importancia de cada variable segun el arbol:")
imp = pd.Series(arbol.feature_importances_, index=Xtr.columns).sort_values(ascending=False)
print(imp.round(3).to_string())

print("\nLas primeras reglas, en texto:")
print(export_text(arbol, feature_names=list(Xtr.columns), max_depth=2))

> **El árbol se explica solo, y sin necesitar escalado.** Sus reglas están en las unidades
> originales: *«si `horas` es menor o igual a 23, entonces…»*. Eso se le puede leer a alguien que no
> sabe de minería y lo entiende.
>
> Es la ventaja que compensa que casi siempre acierte un poco menos.

### ¿Y el KNN?

No tiene coeficientes ni reglas. Para justificar una predicción, **lo único que puede mostrar son
los vecinos que usó**: *«predije esto porque estos cinco registros parecidos tenían estos valores»*.

Eso **no es lo mismo que explicar**. No dice qué variable pesó, ni qué pasaría si el registro fuera
distinto. Es trazabilidad, no interpretabilidad.

---

### La síntesis

| Modelo | ¿Interpretable? | Qué se puede mostrar de una predicción |
|---|---|---|
| **Ridge** | Sí, **si se escaló** | El peso de cada variable, comparable entre sí |
| **Árbol** | Sí, siempre | La cadena de reglas que llevó a esa predicción |
| **KNN** | No | Solo los vecinos usados |

**La pregunta que queda abierta, y que el proyecto tendrá que contestar:** un modelo que acierta más
pero no puede explicarse, ¿sirve? Depende de para qué. Si hay que justificarle una decisión a una
persona —un crédito negado, un diagnóstico, un cupo asignado—, la interpretabilidad **deja de ser un
lujo**.

> **Lo que hace el `Pipeline`:** al llamar `fit`, ajusta cada paso **solo con entrenamiento**. Al
> llamar `score` o `predict` sobre prueba, aplica los parámetros ya aprendidos.
>
> **La fuga se vuelve imposible sin tener que acordarse de nada.** Por eso el proyecto de este curso
> se entrega con `Pipeline`, y no con transformaciones sueltas.

---

## 8. Cómo se decide

```
¿El modelo usa distancias, márgenes, varianza o regularización?
│
├── NO  (árboles, bosques, boosting) ──→ no escalar; no aporta nada
│
└── SÍ
     │
     ├── ¿hay asimetría fuerte? ──→ log primero
     │
     ├── ¿hay atípicos que no se van a quitar? ──→ escalado robusto
     │
     ├── ¿el algoritmo exige un rango acotado? ──→ mín-máx
     │
     └── en cualquier otro caso ──────────────→ puntuación z
```

**Y siempre, sin excepción:** partir primero, `fit` con entrenamiento, `transform` con ambos —
o mejor, `Pipeline`.

---

## 9. Trabajo de la sesión

Sobre **su** dataset:

1. La tabla del aporte a la distancia, **antes** de escalar: qué variable domina y en qué porcentaje.
2. Qué modelo piensan usar, y **si necesita escalado o no**. Una frase justificándolo.
3. Las tres transformaciones aplicadas a la variable de mayor rango, con los cuatro histogramas.
4. **El `Pipeline`** con imputación + escalado, ajustado solo con entrenamiento.
5. Una frase: qué transformación escogieron y por qué — atípicos, asimetría o exigencia del modelo.

---

## Cierre

- **La escala decide quién manda en la distancia.** Sin escalar, un modelo de distancias es un
  modelo de la variable de unidades más grandes.
- **Mín-máx** acota pero es frágil · **z** es el defecto · **robusto** cuando hay atípicos.
- **Escalar no cambia la forma.** Para eso está el logaritmo.
- **La fuga de información no da error, da un resultado mejor de lo real.** `fit` solo con
  entrenamiento.

**Sábado 5:** codificación de categóricas —one-hot, ordinal, alta cardinalidad— y
`ColumnTransformer` para tratar numéricas y categóricas en un mismo flujo. **Cierra el bloque 2.**

**Taller 4** — cierra el **domingo 13 de septiembre**.